# FFHQ100 DAPS Box Inpainting on Colab

This notebook runs DAPS on the 100-image FFHQ benchmark set for **box inpainting**.

Default behavior:

- uses the repo from Google Drive at `/content/drive/MyDrive/DAPS-main`
- uses `data.root=dataset/demo-ffhq`
- runs all 100 PNGs in that folder
- uses the paper-style per-run step budget for this task: `5` ODE / diffusion steps and `200` annealing steps
- saves final metrics to `metrics.json`
- saves per-iteration average PSNR / SSIM / LPIPS trajectories to `metrics_evolution.json`
- disables full trajectory tensor saving to avoid Colab RAM growth

Before running, switch Colab to a GPU runtime with `Runtime -> Change runtime type -> GPU`.


In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import os
import subprocess

# Recommended for this benchmark: use the repo copy on Google Drive.
# Set USE_DRIVE_REPO=False if you prefer cloning from GitHub.
REPO_URL = "https://github.com/Seif-Hussein/daps_test.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/DAPS-main")
USE_DRIVE_REPO = True
DRIVE_REPO_DIR = "/content/drive/MyDrive/DAPS-main"

if USE_DRIVE_REPO:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path(DRIVE_REPO_DIR)
elif not REPO_DIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL, enable USE_DRIVE_REPO, or upload the repo to /content/DAPS-main first."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")
print(f"README exists: {(REPO_DIR / 'README.md').exists()}")

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab-pixel.txt"], check=True)

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import sys
import subprocess
from pathlib import Path

FFHQ_CHECKPOINT_ID = "1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh"
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(exist_ok=True)

ffhq_ckpt = checkpoints_dir / "ffhq256.pt"
if not ffhq_ckpt.exists():
    subprocess.run(
        [sys.executable, "-m", "gdown", f"https://drive.google.com/uc?id={FFHQ_CHECKPOINT_ID}", "-O", str(ffhq_ckpt)],
        check=True,
    )

data_root = Path("dataset/demo-ffhq")
data_files = sorted(data_root.glob("*.png"))
if not data_files:
    raise ValueError("No PNG files found in dataset/demo-ffhq")

print("Checkpoint:", ffhq_ckpt)
print("Dataset root:", data_root)
print("Num images:", len(data_files))

In [ ]:
from pathlib import Path

DATA_CONFIG = "demo-ffhq"
MODEL_CONFIG = "ffhq256ddpm"
TASK_CONFIG = "inpainting"
SAMPLER_CONFIG = "edm_daps"
TASK_GROUP = "pixel"

DATA_ROOT = "dataset/demo-ffhq"
data_files = sorted(Path(DATA_ROOT).glob("*.png"))
if not data_files:
    raise ValueError(f"No PNG files found in {DATA_ROOT}")
NUM_IMAGES = len(data_files)

NUM_RUNS = 1
DIFFUSION_STEPS = 5
ANNEALING_STEPS = 200
BATCH_SIZE = 100
START_ID = 0
END_ID = NUM_IMAGES
RUN_NAME = f"inpainting_box_1k_5x200_ffhq100_benchmark"
SAVE_DIR = "results/pixel/ffhq"
GPU = 0

# Disable full trajectory saving in Colab benchmark runs to avoid linear host RAM growth.
# This keeps progress.json and metrics_evolution.json while skipping xt/x0hat/x0y tensor history.
print(f"Found {NUM_IMAGES} images in {DATA_ROOT}")
print(f"Running slice [{START_ID}:{END_ID}] with batch_size={BATCH_SIZE}")

# Example lighter smoke-test profile:
# NUM_RUNS = 1
# DIFFUSION_STEPS = 5
# ANNEALING_STEPS = 200
# BATCH_SIZE = 5
# START_ID = 0
# END_ID = 10
# RUN_NAME = "phase_retrieval_ffhq100_smoke_test"

In [ ]:
import os
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "posterior_sample.py",
    f"+data={DATA_CONFIG}",
    f"+model={MODEL_CONFIG}",
    f"+task={TASK_CONFIG}",
    f"+sampler={SAMPLER_CONFIG}",
    f"task_group={TASK_GROUP}",
    f"save_dir={SAVE_DIR}",
    f"num_runs={NUM_RUNS}",
    f"sampler.diffusion_scheduler_config.num_steps={DIFFUSION_STEPS}",
    f"sampler.annealing_scheduler_config.num_steps={ANNEALING_STEPS}",
    f"batch_size={BATCH_SIZE}",
    f"data.root={DATA_ROOT}",
    f"data.start_id={START_ID}",
    f"data.end_id={END_ID}",
    f"name={RUN_NAME}",
    f"gpu={GPU}",
    "save_traj=False",
    "save_traj_video=False",
    "save_traj_raw_data=False",
]

print(" \\\n".join(shlex.quote(part) for part in cmd))
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.run(cmd, check=True, env=env)

In [ ]:
import json
from pathlib import Path

result_dir = Path(SAVE_DIR) / RUN_NAME
metrics = json.loads((result_dir / "metrics.json").read_text())
evolution = json.loads((result_dir / "metrics_evolution.json").read_text())

print("Result directory:", result_dir)
print("metrics.json keys:", list(metrics.keys()))
print("metrics_evolution top-level keys:", list(evolution.keys()))

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

result_dir = Path(SAVE_DIR) / RUN_NAME
evolution = json.loads((result_dir / "metrics_evolution.json").read_text())
agg = evolution["aggregate_across_runs"]
iters = agg["iteration"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, metric_name in zip(axes, ["psnr", "ssim", "lpips"]):
    ax.plot(iters, agg["x0y"][metric_name]["mean"], label="x0y")
    ax.plot(iters, agg["x0hat"][metric_name]["mean"], label="x0hat", alpha=0.7)
    ax.set_title(metric_name.upper())
    ax.set_xlabel("Iteration")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import shutil
from pathlib import Path

result_dir = Path(SAVE_DIR) / RUN_NAME
archive_path = shutil.make_archive(str(result_dir), "zip", root_dir=result_dir)
print("Created:", archive_path)